# Lektion 03: Dein erster LLM-Aufruf



| | |
|---|---|
| **Zeit** | ~60 Minuten |
| **Schwierigkeit** | Leicht |
| **Voraussetzungen** | Lektion 01–02 (Funktionen, .env) |
| **Du baust** | Deine erste echte Konversation zwischen Python und einer KI |

## 🎯 Nach dieser Lektion kannst du:

- erklären, was ein LLM ist – und was es NICHT ist

- erklären, was ein API-Call technisch tut (Nachrichten, Rollen, Antwort)

- aus Python eine Nachricht an das konfigurierte Modell senden

- die Antwort korrekt auslesen

- das Verhalten mit System-Nachricht und **Temperature** steuern

## ⚙️ Setup – einmal pro Notebook ausführen

Diese Zelle findet die Projektwurzel und lädt die **zentrale KI-Konfiguration** aus deiner `.env` (siehe `docs/local-ai-server.md`). Du musst nie Notebook-Code ändern, um den Anbieter zu wechseln.

In [ ]:
import sys
from pathlib import Path

# Projektwurzel finden (dort liegt der Ordner "lernakademie")
ROOT = Path.cwd()
while not (ROOT / "lernakademie").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "lernakademie").exists():
    raise RuntimeError("Projektwurzel nicht gefunden – starte das Notebook im Repo-Ordner.")
sys.path.insert(0, str(ROOT))

from lernakademie.config import llm_zusammenfassung

_info = llm_zusammenfassung()  # zeigt NIE den API-Key selbst
print("KI-Konfiguration:", _info["anbieter"], "| Modell:", _info["modell"],
      "| Key:", _info["api_key"])
if _info["anbieter"] == "offline":
    print("Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).")


## 🧠 Konzept: Das LLM – eine sehr belesene, manchmal unzuverlässige Assistentin

**WARUM:** Bis jetzt konnte dein Programm nur das, was du exakt hineingeschrieben hast. Ein **LLM** (Large Language Model, z. B. Mistral) kann **freie Sprache** verstehen und beantworten – das öffnet völlig neue Arten von Programmen: Chatbots, Zusammenfasser, Suchassistenten.

**WAS:** Ein LLM ist eine gigantische **Textfortsetzungs-Maschine**: Es hat aus riesigen Textmengen gelernt, welches Wort wahrscheinlich als Nächstes kommt. Deshalb: Es "weiß" nichts – es **sagt** nur wahrscheinliche, meist sinnvolle Sätze. Manchmal erfindet es überzeugend klingende Dinge (**Halluzination**) – das prägt dein ganzes weiteres Bauen.

**Metapher API:** Stell dir ein **Restaurant** vor: Dein Programm ist der Gast, die **API** ist der Kellner, das Modell ist die Küche. Du gibst dem Kellner eine Bestellung (dein Prompt), die Küche kocht, der Kellner bringt das Gericht (die Antwort). Der **API-Key** ist deine Mitgliedskarte – und `docs/local-ai-server.md` zeigt dir, wie du das ganze Restaurant auch auf **deinen eigenen Rechner** holst (LM Studio, Ollama, …).

## 🧠 Konzept: Nachrichten & Rollen

Ein Gespräch mit einem LLM ist eine **Liste von Nachrichten** (du kennst Listen aus Lektion 01!). Jede Nachricht hat eine **Rolle**:

| Rolle | Wer spricht? | Beispiel |
|---|---|---|
| `system` | Du als Programmierer (Anweisung an den Assistenten) | "Du bist ein freundlicher Erklärbbär." |
| `user` | Der Nutzer | "Was ist eine API?" |
| `assistant` | Das Modell (frühere Antworten) | "Eine API ist …" |

Die System-Nachricht ist wie ein **Flüstern zum Kellner vor Servicebeginn**: "Behandle alle Gäste höflich und erkläre kinderleicht."

## 🔎 Beispiel: Die erste Nachricht

Wir nutzen `lernakademie.llm_client.get_api_client()` – je nach `.env` spricht der Client mit Mistral Cloud, einem lokalen Server **oder** der Offline-Simulation (dann siehst du `[OFFLINE]`-Antworten – der Ablauf ist identisch, nur kostenlos).

In [ ]:
from lernakademie.llm_client import get_api_client, get_model_name

client = get_api_client()
antwort = client.chat.complete(
    model=get_model_name(),
    messages=[
        {"role": "system", "content": "Du bist ein freundlicher Erklaerbaer fuer Technik-Anfaenger."},
        {"role": "user", "content": "Erklaere in zwei Saetzen, was eine API ist."},
    ],
)
print("Rohe Antwort des Servers:", antwort)

### Was sehen wir hier?
Der Server schickt ein Objekt mit `choices` – einer **Liste** von Antwortmöglichkeiten (normalerweise genau einer). Der eigentliche Text steckt in `antwort.choices[0].message.content`. Diese Verschachtelung verstehen zu können, ist dein erste **Datenauslese-Training** – gleich machst du es selbst.

In [ ]:
inhalt = antwort.choices[0].message.content
print("Nur der Text:")
print(inhalt)

## 🧠 Konzept: Temperature – Kreativitäts-Regler

**WAS:** `temperature` steuert, wie **vorhersehbar** das Modell antwortet.
**Metapher:** ein Jazz-Saxofonist: Bei `0.0` spielt er die Noten exakt wie geschrieben. Bei `0.9` improvisiert er frei – meist stimmig, manchmal überraschend.

**Faustregel (merk dir das!):** Fakten & Struktur → niedrig (0.0–0.2) · Ideen & Kreatives → hoch (0.7+).

**TRY:** Führe die Zelle mehrfach aus und vergleiche:

In [ ]:
for temperatur in (0.0, 0.9):
    print(f"--- temperature={temperatur} ---")
    antwort = client.chat.complete(
        model=get_model_name(),
        messages=[{"role": "user",
                   "content": "Nenne ein kreatives Geschenk fuer eine Person, die Kaffee liebt."}],
        temperature=temperatur,
    )
    print(antwort.choices[0].message.content[:200])

## 🧠 Konzept: Tokens & Kosten

LLMs rechnen nicht in Buchstaben oder Wörtern, sondern in **Tokens** (kleine Textstücke, oft 3–4 Zeichen). Faustregel für Deutsch: **`zeichen // 4` ≈ Anzahl Tokens**. Warum das zählt:
- Das **Kontextfenster** (wie viel das Modell auf einmal lesen kann) wird in Tokens gemessen – wie ein Schreibtisch, auf den nur begrenzt Blätter passen.
- Die **Kosten** pro Anfrage zählen Tokens.



In [ ]:
text = "Erklaere in zwei Saetzen, was eine API ist."
print("Geschaetzte Tokens:", len(text) // 4)

## 🛑 Fehler sind Freunde (5): API-Fehler verstehen

Die drei Klassiker und ihre Bedeutung:
- **401 (unauthorized):** Key fehlt oder ist falsch → `.env` prüfen.
- **429 (rate limit):** zu viele Anfragen zu schnell → kurz warten.
- **Connection error:** Server nicht erreichbar → läuft er überhaupt? URL prüfen (`LLM_BASE_URL`).

TRY: Hier simulieren wir einen toten Endpunkt und fangen den Fehler **freundlich** ab:

In [ ]:
import requests

def frage_server(url, frage):
    """Sendet eine Frage an einen OpenAI-kompatiblen Endpoint – mit klaren Fehlermeldungen."""
    try:
        antwort = requests.post(url, json={
            "model": "egal", "messages": [{"role": "user", "content": frage}]},
            timeout=5)
        antwort.raise_for_status()
        return antwort.json()["choices"][0]["message"]["content"]
    except requests.exceptions.ConnectionError:
        return "Verbindung fehlgeschlagen: laeuft der Server? Stimmt die URL (meist mit /v1)?"
    except requests.exceptions.HTTPError as fehler:
        return f"Server-Fehler {fehler.response.status_code}: Key oder Modellname pruefen."

print(frage_server("http://localhost:59999/v1", "Test"))

## ✏️ Deine Aufgabe: Deine eigene Frage-Funktion

Du kapselst den kompletten LLM-Aufruf in EINE wiederverwendbare Funktion – dein erster Baustein für den Knowledge Assistant!

**Anforderungen:**

- Schreibe `frage_modell(frage, temperature=0.0)`, das eine Frage (String) entgegennimmt.

- Die Funktion sendet die Frage an das konfigurierte Modell (mit der Rolle `user`).

- Sie gibt **nur den Antworttext** als String zurück (`return`, nicht `print`).

- Zusatz: bei einem leeren Frage-String gibt sie einen Hinweis-Text zurück (kein Absturz).

**Schreibe deinen Code in die folgende Zelle:**

In [ ]:
from lernakademie.llm_client import get_api_client, get_model_name

def frage_modell(frage, temperature=0.0):
    # YOUR CODE HERE
    pass

print(frage_modell("Was ist Python?"))

### ✅ Richtig, wenn:

- [ ] `frage_modell("...")` gibt einen String zurück (auch im Offline-Modus!).

- [ ] Der Rückgabewert enthält nicht `choices` oder Rohdaten – nur den Text.

- [ ] Ein leerer String stürzt nicht ab.

**🧪 Fuehre die folgende Zelle aus, um dein Ergebnis zu pruefen:**

In [ ]:
ergebnis = frage_modell("Was ist Python?")
assert isinstance(ergebnis, str) and len(ergebnis) > 0, "Es soll ein nicht-leerer String zurueckkommen!"
assert "choices" not in ergebnis, "Nur der Text zurueckgeben, nicht die Rohdaten!"
leer = frage_modell("   ")
assert isinstance(leer, str), "Auch der Leerfall muss einen String liefern."
print("Alle Prüfungen bestanden! ✅")

<details><summary>💡 Hinweis 1 (oeffnen, wenn du nicht weiterkommst)</summary>

Basis: `client = get_api_client()` und dann `client.chat.complete(model=get_model_name(), messages=[...], temperature=temperature)`.
</details>

<details><summary>💡 Hinweis 2 (oeffnen, wenn du nicht weiterkommst)</summary>

Den Text holst du mit `antwort.choices[0].message.content`.
</details>

<details><summary>💡 Hinweis 3 (oeffnen, wenn du nicht weiterkommst)</summary>

Für den Leerfall: `if not frage.strip(): return "Bitte stelle eine echte Frage."` am Anfang.
</details>

## 📝 Reflexion

Beantworte in deinen eigenen Worten:

1. Warum sind die drei Rollen (system/user/assistant) getrennt – was könntest du mit der System-Rolle steuern?

2. Welchen Temperature-Wert würdest du für einen Rechtschreibkorrektor wählen – und warum?

## 🧩 Challenge (freiwillig)

Erweitere `frage_modell` um einen Parameter `system_prompt` (Standard: „Du bist ein hilfreicher Assistent.“) und teste, wie sich die Antwort ändert.

In [ ]:
# YOUR CODE HERE (Challenge)

## ➡️ Weiter

**Weiter mit Lektion 04:** `notebooks/02_llm/04_chatbot_gedaechtnis.ipynb` – aus einem einzelnen Aufruf wird ein **Gespräch mit Gedächtnis**. (Spoiler: Das Gedächtnis ist eine Liste – Lektion 01 lässt grüßen!)